In [ ]:
import numpy as np
import pandas as pd

import tensorflow as tf
import cv2


import matplotlib.pyplot as plt
import seaborn as sns


import os
from pathlib import Path
from PIL import Image 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report , confusion_matrix

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

import os
from tensorflow.keras import layers, models, callbacks



In [ ]:
BASE_DIR = os.path.join("archive (1)", "data")

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VALIDATION_DIR = os.path.join(BASE_DIR, "val")
TEST_DIR = os.path.join(BASE_DIR, "test")

print("TensorFlow:", tf.__version__)
print("Train:", TRAIN_DIR)

In [ ]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 64

# CPU par excessive parallelism avoid karne ke liye
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VALIDATION_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

print("Classes:", train_ds.class_names)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


In [ ]:
model = models.Sequential([

    layers.Input(shape=(64, 64, 3)),

    # Normalize pixels
    layers.Rescaling(1./255),

    # Data augmentation
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),

    # CNN Block 1
    layers.Conv2D(16, 3, activation="relu"),
    layers.MaxPooling2D(),

    # CNN Block 2
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),

    # CNN Block 3
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    # Instead of Flatten -> huge parameters
    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.4),

    # Binary output
    layers.Dense(1, activation="sigmoid")
])


In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
cb = [

    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    ),

    callbacks.ModelCheckpoint(
        "drowsiness_model.keras",
        monitor="val_accuracy",
        save_best_only=True
    )
]


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=cb
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("\nTest Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

In [ ]:



# Predictions
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy().flatten())
    y_pred.extend((predictions.flatten() >= 0.5).astype(int))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Classification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["awake", "sleepy"]
    )
)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["awake", "sleepy"],
    yticklabels=["awake", "sleepy"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Drowsiness Detection Confusion Matrix")
plt.show()

In [ ]:
model.save("drowsiness_model.keras")

print("Model saved successfully!")

In [ ]:
import time

In [ ]:
MODEL_PATH = "drowsiness_model.keras"

if not os.path.exists(MODEL_PATH):
    print("ERROR: drowsiness_model.keras not found!")
    print("Put this file in the same folder as drowsiness_realtime.py")
    exit()

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully!")


In [ ]:
import cv2

print(cv2.__file__)
print(cv2.__version__)
print([x for x in dir(cv2) if "Cascade" in x])

In [ ]:
import sys

print("Python:", sys.executable)

!{sys.executable} -m pip uninstall opencv-python opencv-contrib-python opencv-python-headless -y

In [ ]:
import sys

!{sys.executable} -m pip install --force-reinstall opencv-python==4.10.0.84

In [ ]:
import cv2

print("OpenCV:", cv2.__version__)
print("Location:", cv2.__file__)
print("Cascade available:", hasattr(cv2, "CascadeClassifier"))

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    print("ERROR: Face detector could not be loaded.")
    exit()

print("Face detector loaded!")


In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("ERROR: Could not open webcam.")
    exit()

print("Camera started.")
print("Press Q to quit.")


In [ ]:
sleepy_start_time = None

ALERT_TIME = 2.0

In [ ]:

while True:

    # Capture frame
    ret, frame = cap.read()

    if not ret:
        print("Could not read camera frame.")
        break

In [ ]:
 frame = cv2.flip(frame, 1)

In [ ]:
import cv2

cap = cv2.VideoCapture(0)

print("Camera opened:", cap.isOpened())

ret, frame = cap.read()

print("Frame received:", ret)

if ret:
    print("Frame shape:", frame.shape)

    cv2.imshow("Camera Test", frame)
    cv2.waitKey(3000)
    cv2.destroyAllWindows()
else:
    print("ERROR: Camera frame not received.")

cap.release()

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=42
)

class_names = train_ds.class_names

print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
print("Classes:", class_names)

In [ ]:
MODEL_PATH = "drowsiness_model.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully!")

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_cascade.empty():
    print("ERROR: Face detector could not be loaded.")
else:
    print("Face detector loaded!")

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("ERROR: Could not open webcam.")
    raise SystemExit

print("Camera started!")
print("Press Q to quit.")

sleepy_start_time = None
ALERT_TIME = 2.0

while True:

    ret, frame = cap.read()

    # If one frame fails, don't stop the program
    if not ret:
        print("Frame skipped...")
        continue

    # Mirror camera
    frame = cv2.flip(frame, 1)

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect face
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(80, 80)
    )

    # No face detected
    if len(faces) == 0:
        sleepy_start_time = None

    for (x, y, w, h) in faces:

        # Draw face box
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (255, 0, 0),
            2
        )

        # Crop face
        face = frame[y:y+h, x:x+w]

        # Resize
        face = cv2.resize(face, (64, 64))

        # OpenCV BGR -> RGB
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

        # Float32
        face = np.array(face, dtype=np.float32)

        # Add batch dimension
        face = np.expand_dims(face, axis=0)

        # Prediction
        prediction = model.predict(
            face,
            verbose=0
        )[0][0]

        # Classify
        if prediction >= 0.5:
            label = "Sleepy"
            confidence = prediction
        else:
            label = "Awake"
            confidence = 1 - prediction

        # Display label
        text = f"{label} ({confidence * 100:.1f}%)"

        cv2.putText(
            frame,
            text,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        # -------------------------
        # SLEEPY TIMER
        # -------------------------

        if label == "Sleepy":

            if sleepy_start_time is None:
                sleepy_start_time = time.time()

            elapsed = time.time() - sleepy_start_time

            cv2.putText(
                frame,
                f"Sleepy: {elapsed:.1f}s",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )

            # Alert after 2 seconds
            if elapsed >= ALERT_TIME:

                cv2.putText(
                    frame,
                    "DROWSINESS ALERT!",
                    (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 0, 255),
                    3
                )

        else:
            sleepy_start_time = None

    # Show webcam
    cv2.imshow(
        "DriveSense AI - Drowsiness Detection",
        frame
    )

    # Press Q
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

print("Camera stopped.")